# Fase 6: Framework de Validação Robusta e Backtest Sem Overfitting
### (CPCV, DSR e PBO)

Este notebook executa o protocolo final de validação quantitativa:
1. **Combinatorial Purged Cross-Validation (CPCV)** com Expurgo (*Purging*) e *Embargo*.
2. **Deflated Sharpe Ratio (DSR)** ajustado a assimetria, curtose e múltiplos testes ($N=10$).
3. **Probability of Backtest Overfitting (PBO)** avaliando se a melhor estratégia *In-Sample* mantém desempenho *Out-of-Sample*.

In [1]:
# Cell 1: Importações e Simulação de Matriz de Retornos de Múltiplos Modelos
import os
import sys
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath('..'))

from src.validation.cpcv_evaluator import (
    compute_sharpe_ratio,
    compute_deflated_sharpe_ratio,
    CPCVSplitter,
    compute_pbo_from_cpcv,
    CombinatorialPurgedKFold,
    CPCVEvaluator
)

np.random.seed(42)
n_samples = 1000  # 1000 dias úteis (~4 anos)
n_strategies = 10 # 10 variações do modelo de screening

# Simulação de retornos para 10 variações do modelo
retornos_simulados = np.random.normal(0.0004, 0.012, size=(n_samples, n_strategies))
# Adiciona um ligeiro alpha na estratégia 3 para simular um modelo superior
retornos_simulados[:, 3] += 0.0003
print(f"Matriz de retornos simulada: {retornos_simulados.shape} (amostras x estratégias)")

In [2]:
# Cell 2: Execução do Splitting via CPCV (5 Grupos, 2 de Teste -> 10 Combinações)
cpcv = CPCVSplitter(n_groups=5, k_test_groups=2, purge_window=10, embargo_window=10)

is_sharpes = []
oos_sharpes = []

for fold, (train_idx, test_idx) in enumerate(cpcv.split(n_samples)):
    fold_is_sr = [compute_sharpe_ratio(retornos_simulados[train_idx, s]) for s in range(n_strategies)]
    fold_oos_sr = [compute_sharpe_ratio(retornos_simulados[test_idx, s]) for s in range(n_strategies)]
    
    is_sharpes.append(fold_is_sr)
    oos_sharpes.append(fold_oos_sr)

is_matrix = np.array(is_sharpes)
oos_matrix = np.array(oos_sharpes)
print(f"Combinações CPCV processadas: {is_matrix.shape[0]}")

In [3]:
# Cell 3: Cálculo do DSR e PBO
best_overall_strategy = retornos_simulados[:, 3]
dsr_val = compute_deflated_sharpe_ratio(best_overall_strategy, n_trials=n_strategies)
pbo_val = compute_pbo_from_cpcv(is_matrix, oos_matrix)

print("=== RELATÓRIO DE VALIDAÇÃO DE BACKTEST (FASE 6) ===")
print(f"Número de Combinações CPCV Geradas: {is_matrix.shape[0]}")
print(f"Rácio de Sharpe Anualizado (Melhor Modelo): {compute_sharpe_ratio(best_overall_strategy):.2f}")
print(f"Deflated Sharpe Ratio (DSR) p-value: {dsr_val:.4f} (Validez estatística face a múltiplos testes)")
print(f"Probability of Backtest Overfitting (PBO): {pbo_val * 100:.2f}%")

if pbo_val < 0.30 and dsr_val > 0.95:
    print("\nSTATUS DA APROVAÇÃO: MODELO APROVADO PARA PRODUÇÃO (Baixo risco de overfitting)")
else:
    print("\nSTATUS DA APROVAÇÃO: ALERTA DE OVERFITTING (Rever hiperparâmetros/features)")